# Silver — SC5
Padroniza nomes e tipos da Bronze. Tabelas incrementais (SC5, SC6) recebem MERGE pela chave empresa + R_E_C_N_O_, mantendo a versão de maior S_T_A_M_P_; tabelas FULL (SA1, SA3, SB1) são sobrescritas com o último snapshot de cada empresa. Registros excluídos no Protheus são mantidos com `flg_deletado = true`.

In [ ]:
TABELA = "sc5"

for nome, padrao in [("catalogo_bronze", "dev_bronze"), ("catalogo_silver", "dev_silver"),
                     ("src_path", ""), ("reprocessar_tudo", "false")]:
    dbutils.widgets.text(nome, padrao)

import sys

src_path = dbutils.widgets.get("src_path")
if src_path and src_path not in sys.path:
    sys.path.append(src_path)

In [ ]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

from pdc_lib.transformacoes import TIPOS_CARGA, padronizar, ultima_versao, ultimo_snapshot
from pdc_lib.util import nome_tabela, validar_identificador

catalogo_bronze = validar_identificador(dbutils.widgets.get("catalogo_bronze"))
catalogo_silver = validar_identificador(dbutils.widgets.get("catalogo_silver"))
reprocessar_tudo = dbutils.widgets.get("reprocessar_tudo").lower() == "true"

origem = nome_tabela(catalogo_bronze, "protheus", TABELA)
destino = nome_tabela(catalogo_silver, "protheus", TABELA)

if not spark.catalog.tableExists(origem) or "r_e_c_n_o_" not in spark.table(origem).columns:
    dbutils.notebook.exit(f"Sem dados na Bronze para {TABELA}.")

In [ ]:
bronze = spark.table(origem)
destino_existe = spark.catalog.tableExists(destino)

if TIPOS_CARGA[TABELA] == "FULL":
    snapshot = ultimo_snapshot(padronizar(bronze, TABELA))
    snapshot.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
    print(f"{destino}: sobrescrita com o último snapshot ({snapshot.count()} registros).")
else:
    if destino_existe and not reprocessar_tudo:
        ultimo_lote = spark.table(destino).agg(F.max("_lote")).first()[0]
        if ultimo_lote:
            bronze = bronze.filter(F.col("_lote") > ultimo_lote)
    novos = ultima_versao(padronizar(bronze, TABELA))

    if not destino_existe or reprocessar_tudo:
        novos.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
    else:
        (
            DeltaTable.forName(spark, destino).alias("t")
            .merge(novos.alias("s"), "t.cod_empresa = s.cod_empresa AND t.num_recno = s.num_recno")
            .whenMatchedUpdateAll(condition="s.dat_stamp >= t.dat_stamp")
            .whenNotMatchedInsertAll()
            .execute()
        )
    print(f"{destino}: {novos.count()} registros novos ou alterados processados.")